<a href="https://colab.research.google.com/github/josemanuelmontanoortiz978-droid/PROYECTOS-ED1/blob/main/Ejercicios_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lista Enlazada Simple

In [ ]:
from __future__ import annotations
from typing import Any, Optional


class NodoSimple:
    """Nodo individual para estructuras simplemente enlazadas."""

    def __init__(self, dato: Any, siguiente: Optional[NodoSimple] = None) -> None:
        self.dato: Any = dato
        self.siguiente: Optional[NodoSimple] = siguiente


class ListaSimple:
    """Lista enlazada simple con control de lista vacía y punteros cabeza/cola."""

    def __init__(self) -> None:
        self._cabeza: Optional[NodoSimple] = None
        self._cola: Optional[NodoSimple] = None
        self._tamanio: int = 0

    def esta_vacia(self) -> bool:
        """Verifica si la lista no contiene nodos."""
        return self._cabeza is None

    def insertar_inicio(self, valor: Any) -> None:
        """Inserta un nuevo nodo al inicio en tiempo O(1)."""
        nuevo: NodoSimple = NodoSimple(valor, self._cabeza)
        self._cabeza = nuevo
        if self._cola is None:
            self._cola = nuevo
        self._tamanio += 1

    def insertar_final(self, valor: Any) -> None:
        """Inserta un nuevo nodo al final en tiempo O(1)."""
        nuevo: NodoSimple = NodoSimple(valor)
        if self.esta_vacia():
            self._cabeza = nuevo
            self._cola = nuevo
        else:
            assert self._cola is not None
            self._cola.siguiente = nuevo
            self._cola = nuevo
        self._tamanio += 1

    def buscar(self, valor: Any) -> int:
        """Retorna el índice (0-based) de la primera aparición o -1 si no existe."""
        if self.esta_vacia():
            return -1
        actual: Optional[NodoSimple] = self._cabeza
        indice: int = 0
        while actual is not None:
            if actual.dato == valor:
                return indice
            actual = actual.siguiente
            indice += 1
        return -1

    def eliminar_por_valor(self, valor: Any) -> bool:
        """Elimina la primera ocurrencia de 'valor'. Lanza error si la lista está vacía."""
        if self.esta_vacia():
            raise IndexError("No se puede eliminar: la ListaSimple está vacía.")

        assert self._cabeza is not None
        # Caso 1: El elemento está en la cabeza
        if self._cabeza.dato == valor:
            self._cabeza = self._cabeza.siguiente
            if self._cabeza is None:
                self._cola = None
            self._tamanio -= 1
            return True

        # Caso 2: Buscar el nodo predecesor
        anterior: NodoSimple = self._cabeza
        actual: Optional[NodoSimple] = self._cabeza.siguiente
        while actual is not None:
            if actual.dato == valor:
                anterior.siguiente = actual.siguiente
                if actual == self._cola:
                    self._cola = anterior
                self._tamanio -= 1
                return True
            anterior = actual
            actual = actual.siguiente

        return False

    def imprimir(self) -> str:
        """Recorre e imprime visualmente los enlaces de la lista."""
        if self.esta_vacia():
            representacion = "ListaSimple: [VACÍA]"
            print(representacion)
            return representacion

        elementos: list[str] = []
        actual: Optional[NodoSimple] = self._cabeza
        while actual is not None:
            elementos.append(str(actual.dato))
            actual = actual.siguiente
        representacion = " -> ".join(elementos) + " -> None"
        print(representacion)
        return representacion

    def __len__(self) -> int:
        return self._tamanio


# --- PRUEBAS DE LISTASIMPLE ---
lista = ListaSimple()
lista.imprimir()

lista.insertar_final(20)
lista.insertar_inicio(10)
lista.insertar_final(30)
lista.insertar_final(40)
lista.imprimir()

print("Índice de 30:", lista.buscar(30))
print("Índice de 99:", lista.buscar(99))

lista.eliminar_por_valor(10)  # Elimina cabeza
lista.eliminar_por_valor(40)  # Elimina cola
lista.imprimir()

Pila estática y dinámica

In [ ]:
import ctypes
import time


class PilaEstatica:
    """Pila LIFO sobre bloque contiguo fijo con control de desbordamiento."""

    def __init__(self, capacidad: int) -> None:
        if capacidad <= 0:
            raise ValueError("La capacidad debe ser mayor a 0.")
        self._capacidad: int = capacidad
        self._datos: ctypes.Array[Any] = (ctypes.py_object * capacidad)()
        self._tope: int = -1

    def apilar(self, valor: Any) -> None:
        if self._tope + 1 >= self._capacidad:
            raise OverflowError(f"Stack Overflow: capacidad máxima ({self._capacidad}) alcanzada.")
        self._tope += 1
        self._datos[self._tope] = valor

    def desapilar(self) -> Any:
        if self._tope == -1:
            raise IndexError("Stack Underflow: la pila estática está vacía.")
        valor: Any = self._datos[self._tope]
        self._datos[self._tope] = None
        self._tope -= 1
        return valor

    def esta_vacia(self) -> bool:
        return self._tope == -1


class PilaDinamica:
    """Pila LIFO sobre nodos enlazados sin límite fijo de capacidad."""

    def __init__(self) -> None:
        self._cima: Optional[NodoSimple] = None
        self._tamanio: int = 0

    def apilar(self, valor: Any) -> None:
        self._cima = NodoSimple(valor, self._cima)
        self._tamanio += 1

    def desapilar(self) -> Any:
        if self._cima is None:
            raise IndexError("Stack Underflow: la pila dinámica está vacía.")
        valor: Any = self._cima.dato
        self._cima = self._cima.siguiente
        self._tamanio -= 1
        return valor

    def esta_vacia(self) -> bool:
        return self._cima is None


# 1. Prueba de desbordamiento
p_est = PilaEstatica(capacidad=3)
p_din = PilaDinamica()

for x in [1, 2, 3]:
    p_est.apilar(x)
    p_din.apilar(x)

try:
    p_est.apilar(4)
except OverflowError as err:
    print("Comportamiento PilaEstatica al exceder capacidad:", err)

p_din.apilar(4)
print("Comportamiento PilaDinamica con 4to elemento: aceptado sin límite fijo.")

# 2. Uso intensivo: 100,000 apilados + 100,000 desapilados
N_OPS = 100_000
p_est_bench = PilaEstatica(capacidad=N_OPS)
p_din_bench = PilaDinamica()

t0 = time.perf_counter()
for i in range(N_OPS):
    p_est_bench.apilar(i)
for _ in range(N_OPS):
    p_est_bench.desapilar()
t_estatica = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
for i in range(N_OPS):
    p_din_bench.apilar(i)
for _ in range(N_OPS):
    p_din_bench.desapilar()
t_dinamica = (time.perf_counter() - t0) * 1000

print(f"\nBenchmark ({N_OPS:,} apilar + desapilar):")
print(f" - PilaEstatica (ctypes) : {t_estatica:.2f} ms (sin asignación de objetos en heap)")
print(f" - PilaDinamica (nodos)  : {t_dinamica:.2f} ms (instancia y recolecta {N_OPS:,} nodos)")

Cola circular

In [ ]:
class ColaCircular:
    """Cola FIFO sobre arreglo fijo con aritmética modular para reutilizar espacio."""

    def __init__(self, capacidad: int) -> None:
        if capacidad <= 0:
            raise ValueError("La capacidad debe ser positiva.")
        self._capacidad: int = capacidad
        self._buffer: ctypes.Array[Any] = (ctypes.py_object * capacidad)()
        for i in range(capacidad):
            self._buffer[i] = None
        self._frente: int = 0
        self._final: int = 0
        self._tamanio: int = 0

    def esta_vacia(self) -> bool:
        return self._tamanio == 0

    def esta_llena(self) -> bool:
        return self._tamanio == self._capacidad

    def encolar(self, valor: Any) -> None:
        """Inserta al final envolviendo el índice circularmente en O(1)."""
        if self.esta_llena():
            raise OverflowError("ColaCircular llena: no hay espacios disponibles.")
        self._buffer[self._final] = valor
        self._final = (self._final + 1) % self._capacidad
        self._tamanio += 1

    def desencolar(self) -> Any:
        """Extrae del frente liberando físicamente la celda en O(1)."""
        if self.esta_vacia():
            raise IndexError("ColaCircular vacía.")
        valor: Any = self._buffer[self._frente]
        self._buffer[self._frente] = None
        self._frente = (self._frente + 1) % self._capacidad
        self._tamanio -= 1
        return valor

    def estado_interno(self) -> str:
        """Muestra el contenido físico exacto de cada celda del arreglo."""
        celdas = [self._buffer[i] for i in range(self._capacidad)]
        return f"Buffer={celdas} | frente={self._frente}, final={self._final}, len={self._tamanio}"


# --- DEMOSTRACIÓN DE REUTILIZACIÓN DE ESPACIOS ---
cola_c = ColaCircular(capacidad=4)
for letra in ["A", "B", "C", "D"]:
    cola_c.encolar(letra)
print("1. Llena inicial:       ", cola_c.estado_interno())

print("2. Desencolados:        ", cola_c.desencolar(), "y", cola_c.desencolar())
print("3. Espacios liberados:  ", cola_c.estado_interno())

# Insertamos 'E' y 'F': deben ubicarse en los índices físicos 0 y 1
cola_c.encolar("E")
cola_c.encolar("F")
print("4. Reutilizados [0 y 1]:", cola_c.estado_interno())

Bicola

In [ ]:
class NodoDoble:
    """Nodo con enlaces bidireccionales para estructuras de doble extremo."""

    def __init__(
        self,
        dato: Any,
        anterior: Optional[NodoDoble] = None,
        siguiente: Optional[NodoDoble] = None,
    ) -> None:
        self.dato: Any = dato
        self.anterior: Optional[NodoDoble] = anterior
        self.siguiente: Optional[NodoDoble] = siguiente


class Bicola:
    """Cola de doble extremo (Deque) con operaciones O(1) en frente y final."""

    def __init__(self) -> None:
        self._frente: Optional[NodoDoble] = None
        self._final: Optional[NodoDoble] = None
        self._tamanio: int = 0

    def esta_vacia(self) -> bool:
        return self._tamanio == 0

    def agregar_frente(self, valor: Any) -> None:
        nuevo = NodoDoble(valor, anterior=None, siguiente=self._frente)
        if self.esta_vacia():
            self._frente = self._final = nuevo
        else:
            assert self._frente is not None
            self._frente.anterior = nuevo
            self._frente = nuevo
        self._tamanio += 1

    def agregar_final(self, valor: Any) -> None:
        nuevo = NodoDoble(valor, anterior=self._final, siguiente=None)
        if self.esta_vacia():
            self._frente = self._final = nuevo
        else:
            assert self._final is not None
            self._final.siguiente = nuevo
            self._final = nuevo
        self._tamanio += 1

    def eliminar_frente(self) -> Any:
        if self.esta_vacia():
            raise IndexError("La bicola está vacía.")
        assert self._frente is not None
        valor = self._frente.dato
        self._frente = self._frente.siguiente
        if self._frente is None:
            self._final = None
        else:
            self._frente.anterior = None
        self._tamanio -= 1
        return valor

    def eliminar_final(self) -> Any:
        if self.esta_vacia():
            raise IndexError("La bicola está vacía.")
        assert self._final is not None
        valor = self._final.dato
        self._final = self._final.anterior
        if self._final is None:
            self._frente = None
        else:
            self._final.siguiente = None
        self._tamanio -= 1
        return valor

    def __len__(self) -> int:
        return self._tamanio


# --- SIMULACIÓN DE PILA (LIFO) Y COLA (FIFO) CON LA MISMA BICOLA ---
# 1. Simulación de Pila: entrar y salir por el MISMO extremo (final / final)
bicola_pila = Bicola()
for paso in ["Paso 1", "Paso 2", "Paso 3"]:
    bicola_pila.agregar_final(paso)  # Push

salida_lifo = []
while not bicola_pila.esta_vacia():
    salida_lifo.append(bicola_pila.eliminar_final())  # Pop
print("Simulación Pila (LIFO) con Bicola:", salida_lifo)

# 2. Simulación de Cola: entrar por un extremo (final) y salir por el opuesto (frente)
bicola_cola = Bicola()
for cliente in ["Cliente 1", "Cliente 2", "Cliente 3"]:
    bicola_cola.agregar_final(cliente)  # Enqueue

salida_fifo = []
while not bicola_cola.esta_vacia():
    salida_fifo.append(bicola_cola.eliminar_frente())  # Dequeue
print("Simulación Cola (FIFO) con Bicola:", salida_fifo)

Tabla hash

In [ ]:
class NodoHash:
    """Nodo par clave-valor para resolución de colisiones por encadenamiento."""

    def __init__(self, clave: str, valor: Any, siguiente: Optional[NodoHash] = None) -> None:
        self.clave: str = clave
        self.valor: Any = valor
        self.siguiente: Optional[NodoHash] = siguiente


class TablaHash:
    """Tabla Hash con encadenamiento separado implementada desde cero."""

    def __init__(self, capacidad: int = 16) -> None:
        if capacidad <= 0:
            raise ValueError("La capacidad de la tabla debe ser mayor a 0.")
        self._capacidad: int = capacidad
        self._cubetas: ctypes.Array[Any] = (ctypes.py_object * capacidad)()
        for i in range(capacidad):
            self._cubetas[i] = None
        self._tamanio: int = 0

    def _hash(self, clave: str) -> int:
        """Función hash polinómica determinista (base 31) proyectada al rango de cubetas."""
        acumulador: int = 0
        for caracter in clave:
            acumulador = (acumulador * 31 + ord(caracter)) % self._capacidad
        return acumulador

    def insertar(self, clave: str, valor: Any) -> None:
        """Inserta un par clave-valor o actualiza el valor si la clave ya existe."""
        indice: int = self._hash(clave)
        actual: Optional[NodoHash] = self._cubetas[indice]

        # Verificar si la clave ya existe para actualizarla en sitio
        while actual is not None:
            if actual.clave == clave:
                actual.valor = valor
                return
            actual = actual.siguiente

        # Clave nueva: insertar al inicio de la cadena de la cubeta en O(1)
        self._cubetas[indice] = NodoHash(clave, valor, self._cubetas[indice])
        self._tamanio += 1

    def buscar(self, clave: str) -> Any:
        """Retorna el valor asociado a 'clave' o lanza KeyError si no existe."""
        indice: int = self._hash(clave)
        actual: Optional[NodoHash] = self._cubetas[indice]
        while actual is not None:
            if actual.clave == clave:
                return actual.valor
            actual = actual.siguiente
        raise KeyError(f"Clave no encontrada en TablaHash: '{clave}'")

    def eliminar(self, clave: str) -> Any:
        """Elimina la entrada por clave y retorna su valor."""
        indice: int = self._hash(clave)
        anterior: Optional[NodoHash] = None
        actual: Optional[NodoHash] = self._cubetas[indice]

        while actual is not None:
            if actual.clave == clave:
                if anterior is None:
                    self._cubetas[indice] = actual.siguiente
                else:
                    anterior.siguiente = actual.siguiente
                self._tamanio -= 1
                return actual.valor
            anterior = actual
            actual = actual.siguiente

        raise KeyError(f"No se puede eliminar; clave inexistente: '{clave}'")

    def items(self) -> list[tuple[str, Any]]:
        """Retorna todos los pares (clave, valor) almacenados en la tabla."""
        pares: list[tuple[str, Any]] = []
        for i in range(self._capacidad):
            actual: Optional[NodoHash] = self._cubetas[i]
            while actual is not None:
                pares.append((actual.clave, actual.valor))
                actual = actual.siguiente
        return pares

    def __len__(self) -> int:
        return self._tamanio


# --- PRUEBAS DE TABLAHASH ---
tabla = TablaHash(capacidad=4)  # Capacidad pequeña para forzar colisiones
tabla.insertar("USR-01", "Ana")
tabla.insertar("USR-02", "Luis")
tabla.insertar("USR-03", "Carla")
tabla.insertar("USR-04", "Mateo")
tabla.insertar("USR-05", "Sofía")

# Actualización de clave existente
tabla.insertar("USR-02", "Luis Alberto (Actualizado)")
print("Búsqueda USR-02:", tabla.buscar("USR-02"))
print("Total elementos:", len(tabla))

tabla.eliminar("USR-03")
print("Total tras eliminar USR-03:", len(tabla))

Desafío

In [ ]:
import json
from dataclasses import dataclass, asdict


@dataclass
class TicketTurno:
    """Representa un turno de atención registrado en el sistema."""
    codigo: str
    cliente: str
    tramite: str
    estado: str = "EN ESPERA"


class SistemaTurnos:
    """Gestiona turnos combinando ColaCircular (orden FIFO) y TablaHash (búsqueda O(1))."""

    def __init__(self, capacidad_sala: int = 20) -> None:
        self._cola_espera: ColaCircular = ColaCircular(capacidad=capacidad_sala)
        self._indice_tickets: TablaHash = TablaHash(capacidad=capacidad_sala * 2)
        self._correlativo: int = 100

    def emitir_turno(self, cliente: str, tramite: str) -> TicketTurno:
        """Crea un nuevo turno, lo encola para atención y lo indexa en la TablaHash."""
        if self._cola_espera.esta_llena():
            raise OverflowError("Sala de espera llena. No se pueden emitir más turnos.")
        self._correlativo += 1
        codigo: str = f"T-{self._correlativo}"
        ticket: TicketTurno = TicketTurno(codigo=codigo, cliente=cliente, tramite=tramite)

        self._cola_espera.encolar(codigo)
        self._indice_tickets.insertar(codigo, ticket)
        return ticket

    def llamar_siguiente(self) -> Optional[TicketTurno]:
        """Desencola el siguiente código en FIFO y actualiza su estado en la TablaHash."""
        if self._cola_espera.esta_vacia():
            return None
        codigo_siguiente: str = self._cola_espera.desencolar()
        ticket: TicketTurno = self._indice_tickets.buscar(codigo_siguiente)
        ticket.estado = "ATENDIDO"
        self._indice_tickets.insertar(codigo_siguiente, ticket)
        return ticket

    def consultar_turno(self, codigo: str) -> TicketTurno:
        """Búsqueda directa O(1) promedio por código de ticket."""
        return self._indice_tickets.buscar(codigo)

    def finalizar_jornada(self, ruta_json: Optional[str] = None) -> list[dict[str, Any]]:
        """Genera el reporte de cierre y persiste opcionalmente en disco en formato JSON."""
        registros: list[dict[str, Any]] = [
            asdict(ticket) for _, ticket in self._indice_tickets.items()
        ]
        registros.sort(key=lambda item: item["codigo"])

        if ruta_json is not None:
            with open(ruta_json, "w", encoding="utf-8") as archivo:
                json.dump(registros, archivo, indent=2, ensure_ascii=False)
            print(f"Historial persistido correctamente en '{ruta_json}'.")

        return registros


# --- SIMULACIÓN COMPLETA DEL SISTEMA DE TURNOS ---
centro_atencion = SistemaTurnos(capacidad_sala=5)

t1 = centro_atencion.emitir_turno("Valeria Mendoza", "Apertura de Cuenta")
t2 = centro_atencion.emitir_turno("Diego Rojas", "Soporte Técnico")
t3 = centro_atencion.emitir_turno("Camila Ortiz", "Pagos y Facturación")

print("Turno llamado a ventanilla:", centro_atencion.llamar_siguiente())
print("Consulta rápida T-101:", centro_atencion.consultar_turno("T-101"))
print("Consulta rápida T-102:", centro_atencion.consultar_turno("T-102"))

# Cierre de jornada con persistencia opcional activada
reporte_final = centro_atencion.finalizar_jornada(ruta_json="cierre_turnos.json")
print("Resumen final del día:", reporte_final)